# DUKASCOPY

### IMPORTS

In [ ]:
!pip install dukascopy-python pandas pyarrow joblib -q

In [ ]:
import os
import time
import pandas as pd
import dukascopy_python as dp
from datetime import datetime
from pathlib import Path
from joblib import Parallel, delayed

### PARAMS

In [ ]:
# Number of parallel jobs to run
N_JOBS = 4 

# Currency pairs and months to download
PAIRS = ["USD/ZAR"]

# Months to download in YYYYMM format
MONTHS = [
    "202401"
]

### GOOGLE DRIVE

In [ ]:
from google.colab import drive

# Mount Google Drive to save the downloaded data
drive.mount('/content/drive')

# Output directory for processed data
OUT_DIR = Path("/content/drive/MyDrive/GITHUB-COPILOT/stk-mat2011/data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

### GET MONTH RANGE

In [ ]:
# Helper function to get start and end datetime for a given month
def get_month_range(yyyymm: str):
    """
    Given a month in YYYYMM format, return the start and end datetime objects for that month.
    """
    year, month = int(yyyymm[:4]), int(yyyymm[4:6])
    start = datetime(year, month, 1)
    end = datetime(year + 1, 1, 1) if month == 12 else datetime(year, month + 1, 1)
    return start, end

### PROCESS JOB

In [ ]:
# Main function to process each job with existence check and retry logic
def process_job(pair, yyyymm, compression="snappy", max_retries=3):
    """
    Function to be executed in parallel with existence check and retries
    """
    symbol = pair.replace("/", "").lower()
    
    # 1. EXISTENCE CHECK: Define file paths first
    out_name_bid = f"{symbol}_dukascopy_bid_{yyyymm}.parquet"
    out_name_ask = f"{symbol}_dukascopy_ask_{yyyymm}.parquet"
    out_path_bid = OUT_DIR / out_name_bid
    out_path_ask = OUT_DIR / out_name_ask

    # If both files already exist, skip the download completely
    if out_path_bid.exists() and out_path_ask.exists():
        return f"SKIPPED (ALREADY EXISTS): {pair} {yyyymm}"

    start, end = get_month_range(yyyymm)
    
    # 2. RETRY LOGIC: Handle Dukascopy connection drops
    for attempt in range(max_retries):
        try:
            df = dp.fetch(
                instrument=pair,
                interval=dp.INTERVAL_TICK,
                offer_side=dp.OFFER_SIDE_BID, 
                start=start,
                end=end,
            )

            if df is None or len(df) == 0:
                return f"EMPTY: {pair} {yyyymm}"

            results_stats = []
            for side, price_col, vol_col in [("bid", "bidPrice", "bidVolume"), ("ask", "askPrice", "askVolume")]:
                out_name = out_name_bid if side == "bid" else out_name_ask
                out_path = out_path_bid if side == "bid" else out_path_ask
                
                pd.DataFrame({
                    "datetime": df.index,
                    "symbol": symbol.upper(),
                    "price_type": side.upper(),
                    "price": df[price_col].values,
                    "volume": df[vol_col].values,
                }).to_parquet(out_path, engine="pyarrow", compression=compression, index=False)
                
                results_stats.append(f"{out_name} ({len(df):,} rows)")
                
            return f"SUCCESS: {pair} {yyyymm} -> " + " & ".join(results_stats)
        
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(5) # Wait 5 seconds before retrying
                continue
            return f"FAILED: {pair} {yyyymm} | Error after {max_retries} attempts: {str(e)}"


### EXECUTION

In [ ]:
# Create list of jobs (pair, month) and run in parallel
jobs = [(p, m) for p in PAIRS for m in MONTHS]

print(f"Downloading {len(jobs)} jobs · {N_JOBS} workers\n")

# Run the jobs in parallel and print results
t0 = time.time()
results = Parallel(n_jobs=N_JOBS, backend="threading")(
    delayed(process_job)(p, m) for p, m in jobs
)

# Print results
for r in results:
    print(f"  {r}")

# Final stats
elapsed = time.time() - t0

# Print final statistics
print(f"\nDone in {elapsed:.1f}s")
print(f"Saved to {OUT_DIR.absolute()}")